In [8]:
"""
DigitaltMuseum Explorer — Bohusläns Museum & Lödöse Museum
==========================================================
Focused exploration of what's actually in the two confirmed
collections on DigitaltMuseum, to understand the data before
building a cross-museum knowledge graph.

Prerequisites:
    pip install requests pandas

Usage:
    python dimu_explorer.py
    — or import functions into a notebook/REPL

API key: "demo" returns max 10 rows per query.
For serious work, contact support@kulturit.no for a full key.
"""

import requests
import json
import time
from collections import Counter
from pprint import pprint

# ============================================================
# CONFIG
# ============================================================

DIMU_BASE = "https://api.dimu.org/api/solr/select"
DIMU_API_KEY = "demo"  # Max 10 rows. Get a real key for bulk work.

MUSEUMS = {
    "bohuslan": {"name": "Bohusläns museum", "code": "S-BM"},
    "lodose":   {"name": "Lödöse museum",    "code": "S-LM"},
}


# ============================================================
# CORE API FUNCTIONS
# ============================================================

def dimu_query(owner, q="*", rows=10, start=0, fq_extra=None,
               facet_field=None, facet_limit=50, fl=None):
    """
    Flexible DigitaltMuseum query.
    
    Parameters:
        owner       — museum code (e.g. "S-BM")
        q           — search query (* for all)
        rows        — results per page (max 100, demo key caps at 10)
        start       — pagination offset
        fq_extra    — additional filter query string(s), str or list
        facet_field — field to facet on (returns counts instead of docs)
        facet_limit — max facet values to return
        fl          — field list to return (comma-separated string)
    
    Returns parsed JSON.
    """
    params = {
        "q": q,
        "fq": f"identifier.owner:{owner}*",
        "wt": "json",
        "rows": rows,
        "start": start,
        "api.key": DIMU_API_KEY,
    }
    
    if fl:
        params["fl"] = fl
    
    if facet_field:
        params["facet"] = "true"
        params["facet.field"] = facet_field
        params["facet.limit"] = facet_limit
        params["facet.mincount"] = 1
        params["rows"] = 0  # Don't need docs for facet queries
    
    # Handle extra filter queries
    if fq_extra:
        if isinstance(fq_extra, str):
            fq_extra = [fq_extra]
        # Solr supports multiple fq params — we need to send them separately
        # requests handles this if we use a list of tuples
        param_tuples = list(params.items())
        for fq in fq_extra:
            param_tuples.append(("fq", fq))
        del params  # avoid confusion
        r = requests.get(DIMU_BASE, params=param_tuples, timeout=30)
    else:
        r = requests.get(DIMU_BASE, params=params, timeout=30)
    
    r.raise_for_status()
    return r.json()


def count(owner, q="*", fq_extra=None):
    """Quick count of matching objects."""
    result = dimu_query(owner, q=q, rows=0, fq_extra=fq_extra)
    return result.get("response", {}).get("numFound", 0)


def facet(owner, field, q="*", limit=30, fq_extra=None):
    """
    Get faceted value counts for a field.
    Returns list of (value, count) tuples.
    """
    result = dimu_query(owner, q=q, facet_field=field,
                        facet_limit=limit, fq_extra=fq_extra)
    
    raw = (result.get("facet_counts", {})
                 .get("facet_fields", {})
                 .get(field, []))
    
    pairs = []
    for i in range(0, len(raw), 2):
        pairs.append((raw[i], raw[i + 1]))
    return pairs


def sample(owner, q="*", n=10, fq_extra=None):
    """
    Fetch sample documents. Returns the docs list.
    """
    result = dimu_query(owner, q=q, rows=n, fq_extra=fq_extra)
    return result.get("response", {}).get("docs", [])


def inspect_fields(doc, indent=2):
    """Pretty-print a single document's fields."""
    prefix = " " * indent
    for k, v in sorted(doc.items()):
        val_str = str(v)
        if len(val_str) > 100:
            val_str = val_str[:100] + "..."
        print(f"{prefix}{k:45s}  {val_str}")


# ============================================================
# EXPLORATION STEPS
# ============================================================

def step1_collection_overview():
    """
    What does each museum have? Object types, rough scale,
    and whether images/geodata are common.
    """
    print("=" * 65)
    print("STEP 1: Collection overview")
    print("=" * 65)
    
    for key, m in MUSEUMS.items():
        total = count(m["code"])
        with_pics = count(m["code"], fq_extra="artifact.hasPictures:true")
        
        print(f"\n{'─' * 50}")
        print(f"  {m['name']} ({m['code']})")
        print(f"  Total objects:      {total:>10,}")
        print(f"  With images:        {with_pics:>10,}")
        
        # Object types
        types = facet(m["code"], "artifact.type", limit=15)
        if types:
            print(f"\n  Object types:")
            for name, cnt in types:
                bar = "█" * min(cnt // max(total // 200, 1), 40)
                print(f"    {name:30s} {cnt:>8,}  {bar}")
        
        time.sleep(0.5)


def step2_inspect_sample_records():
    """
    Pull a few actual records from each museum and list ALL fields.
    This is the key step — you need to see what metadata is available
    before you can design your graph schema.
    """
    print("\n" + "=" * 65)
    print("STEP 2: Sample record fields")
    print("=" * 65)
    
    for key, m in MUSEUMS.items():
        print(f"\n{'─' * 50}")
        print(f"  {m['name']} — sample records")
        print(f"{'─' * 50}")
        
        docs = sample(m["code"], n=3)
        
        if not docs:
            print("  (no results — demo key limitation?)")
            continue
        
        for i, doc in enumerate(docs):
            print(f"\n  ── Record {i+1} ──")
            inspect_fields(doc)
    
        # Collect all field names across a batch
        all_docs = sample(m["code"], n=10)
        all_fields = set()
        for d in all_docs:
            all_fields.update(d.keys())
        
        print(f"\n  All fields observed ({len(all_fields)}):")
        for f in sorted(all_fields):
            print(f"    {f}")


def step3_what_connects_them():
    """
    Look for thematic and categorical overlaps between the two museums.
    These become the candidate edges in a cross-museum graph.
    """
    print("\n" + "=" * 65)
    print("STEP 3: What connects Bohusläns museum and Lödöse museum?")
    print("=" * 65)
    
    # --- Object classifications ---
    print("\n  A) Object name/classification overlap")
    print("  " + "─" * 45)
    
    bm_names = dict(facet("S-BM", "artifact.name", limit=50))
    lm_names = dict(facet("S-LM", "artifact.name", limit=50))
    
    shared_names = set(bm_names.keys()) & set(lm_names.keys())
    if shared_names:
        print(f"  Shared classifications ({len(shared_names)}):")
        for name in sorted(shared_names):
            print(f"    {name:30s}  BM: {bm_names[name]:>6,}  LM: {lm_names[name]:>6,}")
    else:
        print("  No shared classifications in top 50 of each.")
    
    # --- Materials ---
    print("\n  B) Material overlap")
    print("  " + "─" * 45)
    
    bm_mat = dict(facet("S-BM", "artifact.material", limit=50))
    lm_mat = dict(facet("S-LM", "artifact.material", limit=50))
    
    shared_mat = set(bm_mat.keys()) & set(lm_mat.keys())
    if shared_mat:
        print(f"  Shared materials ({len(shared_mat)}):")
        for name in sorted(shared_mat):
            print(f"    {name:30s}  BM: {bm_mat[name]:>6,}  LM: {lm_mat[name]:>6,}")
    else:
        print("  No shared materials in top 50 of each.")
    
    # --- Places ---
    print("\n  C) Place name overlap")
    print("  " + "─" * 45)
    
    bm_places = dict(facet("S-BM", "artifact.event.place", limit=50))
    lm_places = dict(facet("S-LM", "artifact.event.place", limit=50))
    
    shared_places = set(bm_places.keys()) & set(lm_places.keys())
    if shared_places:
        print(f"  Shared places ({len(shared_places)}):")
        for name in sorted(shared_places):
            print(f"    {name:30s}  BM: {bm_places[name]:>6,}  LM: {lm_places[name]:>6,}")
    else:
        print("  No shared places in top 50 of each.")
    
    # --- Producers/creators ---
    print("\n  D) Producer/creator overlap")
    print("  " + "─" * 45)
    
    bm_prod = dict(facet("S-BM", "artifact.ingress.producer", limit=50))
    lm_prod = dict(facet("S-LM", "artifact.ingress.producer", limit=50))
    
    shared_prod = set(bm_prod.keys()) & set(lm_prod.keys())
    if shared_prod:
        print(f"  Shared producers ({len(shared_prod)}):")
        for name in sorted(shared_prod):
            print(f"    {name:30s}  BM: {bm_prod[name]:>6,}  LM: {lm_prod[name]:>6,}")
    else:
        print("  No shared producers in top 50 of each.")
    
    # --- Text searches for connecting themes ---
    print("\n  E) Thematic keyword counts")
    print("  " + "─" * 45)
    
    keywords = [
        "medeltid",       # medieval
        "vikingatid",     # viking age
        "arkeologi",      # archaeology
        "keramik",        # ceramics
        "Göta älv",       # the river
        "Lödöse",         # the town
        "Bohuslän",       # the region
        "handel",         # trade
        "sjöfart",        # seafaring
        "kyrka",          # church
        "begravning",     # burial
        "mynt",           # coins
        "ben",            # bone
        "flinta",         # flint
        "runsten",        # runestone
    ]
    
    print(f"  {'Keyword':20s} {'Bohusläns':>10s} {'Lödöse':>10s}")
    print(f"  {'─'*20} {'─'*10} {'─'*10}")
    
    for kw in keywords:
        bm_count = count("S-BM", q=kw)
        lm_count = count("S-LM", q=kw)
        if bm_count > 0 or lm_count > 0:
            print(f"  {kw:20s} {bm_count:>10,} {lm_count:>10,}")
        time.sleep(0.3)


def step4_time_periods():
    """
    Explore temporal distribution. What time periods are represented?
    This helps identify where the collections overlap chronologically.
    """
    print("\n" + "=" * 65)
    print("STEP 4: Temporal distribution")
    print("=" * 65)
    
    # The dating fields vary — let's check what's available
    for key, m in MUSEUMS.items():
        print(f"\n  {m['name']}:")
        
        # Try different time-related facets
        time_fields = [
            "artifact.event.timespan.fromYear",
            "artifact.ingress.production_date",
            "artifact.time",
        ]
        
        for tf in time_fields:
            values = facet(m["code"], tf, limit=20)
            if values:
                print(f"\n    Field: {tf}")
                for val, cnt in values[:15]:
                    print(f"      {val:30s} {cnt:>6,}")
            time.sleep(0.3)


def step5_deep_dive_lodose():
    """
    Lödöse is the smaller, more focused collection — mostly medieval
    archaeology. Let's see what's there in detail.
    """
    print("\n" + "=" * 65)
    print("STEP 5: Deep dive into Lödöse museum")
    print("=" * 65)
    
    total = count("S-LM")
    print(f"  Total objects: {total:,}")
    
    # What types of objects?
    print("\n  Object types:")
    for name, cnt in facet("S-LM", "artifact.type", limit=20):
        pct = cnt / total * 100 if total > 0 else 0
        print(f"    {name:30s} {cnt:>6,}  ({pct:5.1f}%)")
    
    # Object names/classifications
    print("\n  Top classifications:")
    for name, cnt in facet("S-LM", "artifact.name", limit=25):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Materials
    print("\n  Top materials:")
    for name, cnt in facet("S-LM", "artifact.material", limit=20):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Places
    print("\n  Places mentioned:")
    for name, cnt in facet("S-LM", "artifact.event.place", limit=20):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Techniques
    print("\n  Techniques:")
    for name, cnt in facet("S-LM", "artifact.technique", limit=15):
        print(f"    {name:30s} {cnt:>6,}")


def step6_deep_dive_bohuslan():
    """
    Bohusläns museum is much larger — 327k objects spanning coastal
    culture, archaeology, natural history, textiles, and more.
    """
    print("\n" + "=" * 65)
    print("STEP 6: Deep dive into Bohusläns museum")
    print("=" * 65)
    
    total = count("S-BM")
    print(f"  Total objects: {total:,}")
    
    # What types of objects?
    print("\n  Object types:")
    for name, cnt in facet("S-BM", "artifact.type", limit=15):
        pct = cnt / total * 100 if total > 0 else 0
        print(f"    {name:30s} {cnt:>6,}  ({pct:5.1f}%)")
    
    # Object names/classifications
    print("\n  Top classifications:")
    for name, cnt in facet("S-BM", "artifact.name", limit=25):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Materials
    print("\n  Top materials:")
    for name, cnt in facet("S-BM", "artifact.material", limit=20):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Places
    print("\n  Top places:")
    for name, cnt in facet("S-BM", "artifact.event.place", limit=25):
        print(f"    {name:30s} {cnt:>6,}")
    
    # Producers
    print("\n  Top producers/creators:")
    for name, cnt in facet("S-BM", "artifact.ingress.producer", limit=15):
        print(f"    {name:30s} {cnt:>6,}")


def step7_toward_the_graph():
    """
    Based on what we've found, outline the graph schema
    and pull a concrete batch of records for initial graph construction.
    
    This doesn't build the graph (you'll do that in NetworkX/Neo4j),
    but it structures the data extraction.
    """
    print("\n" + "=" * 65)
    print("STEP 7: Data extraction for graph building")
    print("=" * 65)
    
    print("""
    Proposed graph schema (based on exploration):
    
    NODE TYPES:
      Museum         — the two institutions
      Object         — individual artifacts/photos
      Place          — geographic locations
      Material       — materials (keramik, ben, järn, etc.)
      Classification — object types/names (krukskärva, nål, etc.)
      TimePeriod     — temporal markers
      Producer       — creators/makers (where known)
    
    EDGE TYPES:
      (:Museum)-[:HAS_OBJECT]->(:Object)
      (:Object)-[:FOUND_AT]->(:Place)
      (:Object)-[:MADE_OF]->(:Material)
      (:Object)-[:CLASSIFIED_AS]->(:Classification)
      (:Object)-[:DATED_TO]->(:TimePeriod)
      (:Object)-[:CREATED_BY]->(:Producer)
      
    CROSS-MUSEUM EDGES (implicit, via shared nodes):
      Objects from both museums sharing the same Place
      Objects from both museums sharing the same Material
      Objects from both museums sharing the same Classification
      Objects from both museums in the same TimePeriod
    """)
    
    # Pull a batch of records from each museum for graph construction
    print("  Pulling sample records for graph construction...")
    print("  (With demo key, limited to 10 per query)\n")
    
    all_records = []
    
    for key, m in MUSEUMS.items():
        docs = sample(m["code"], n=10)
        print(f"  {m['name']}: {len(docs)} records retrieved")
        
        for doc in docs:
            record = {
                "museum": key,
                "museum_name": m["name"],
                "id": doc.get("identifier.id", ""),
                "uuid": doc.get("uuid", ""),
                "type": doc.get("artifact.type", ""),
                "name": doc.get("artifact.name", ""),
                "title": doc.get("artifact.ingress.title", ""),
                "producer": doc.get("artifact.ingress.producer", ""),
                "production_date": doc.get("artifact.ingress.production_date", ""),
                "material": doc.get("artifact.material", ""),
                "technique": doc.get("artifact.technique", ""),
                "place": doc.get("artifact.event.place", ""),
                "description": doc.get("artifact.ingress.description", ""),
                "has_pictures": doc.get("artifact.hasPictures", False),
            }
            all_records.append(record)
        
        time.sleep(0.5)
    
    # Show extracted records
    print(f"\n  Total extracted: {len(all_records)} records")
    print(f"\n  Sample extracted record:")
    if all_records:
        for k, v in all_records[0].items():
            if v:
                val = str(v)[:80]
                print(f"    {k:20s}: {val}")
    
    # Show which fields are populated
    print(f"\n  Field coverage across {len(all_records)} records:")
    for field in ["type", "name", "producer", "production_date",
                  "material", "technique", "place", "description"]:
        filled = sum(1 for r in all_records if r.get(field))
        pct = filled / len(all_records) * 100 if all_records else 0
        bar = "█" * int(pct / 2.5)
        print(f"    {field:20s} {filled:>3}/{len(all_records)}  ({pct:5.1f}%)  {bar}")
    
    return all_records


# ============================================================
# BONUS: K-SAMSÖK NAME HUNTING
# ============================================================

def hunt_ksamsok_names():
    """
    Try various serviceOrganization and serviceName values
    to find if/how these museums appear in K-samsök.
    
    The zero results earlier might be due to:
    - Different exact naming (case, punctuation)
    - Delivery through Västarvet (regional umbrella)
    - Museums not currently delivering to K-samsök
    """
    KSAMSOK_BASE = "https://kulturarvsdata.se/ksamsok/api"
    
    print("\n" + "=" * 65)
    print("BONUS: Hunting for museums in K-samsök")
    print("=" * 65)
    
    # Try different name variants
    queries = [
        # serviceOrganization variants
        'serviceOrganization="Bohusläns museum"',
        'serviceOrganization="Bohusläns Museum"',
        'serviceOrganization="bohusläns museum"',
        'serviceOrganization="Västarvet"',
        'serviceOrganization="Lödöse museum"',
        'serviceOrganization="Lödöse Museum"',
        'serviceOrganization="Göteborgs stadsmuseum"',
        'serviceOrganization="Göteborgs Stadsmuseum"',
        # serviceName variants (dataset identifiers)
        'serviceName="bohm"',
        'serviceName="BohM"',
        'serviceName="lod"',
        'serviceName="gsm"',
        'serviceName="GSM"',
        # Text search for the museum names
        'text="Bohusläns museum"',
        'text="Lödöse museum"',
    ]
    
    for cql in queries:
        try:
            params = {
                "method": "search",
                "version": "1.1",
                "query": cql,
                "startRecord": 1,
                "hitsPerPage": 1,
                "recordSchema": "presentation",
            }
            r = requests.get(KSAMSOK_BASE, params=params, timeout=15)
            r.raise_for_status()
            
            import xml.etree.ElementTree as ET
            root = ET.fromstring(r.text)
            ns = {"srw": "http://www.loc.gov/zing/srw/"}
            total_el = root.find(".//srw:numberOfRecords", ns)
            total = int(total_el.text) if total_el is not None else 0
            
            status = f"{total:>8,}" if total > 0 else "       0"
            marker = " ◄── HIT!" if total > 0 else ""
            print(f"  {cql:55s} → {status}{marker}")
            
        except Exception as e:
            print(f"  {cql:55s} → Error: {e}")
        
        time.sleep(0.5)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("╔═════════════════════════════════════════════════════════════╗")
    print("║  DigitaltMuseum Explorer                                   ║")
    print("║  Bohusläns Museum (S-BM) · Lödöse Museum (S-LM)          ║")
    print("╚═════════════════════════════════════════════════════════════╝")
    print()
    print("Available steps:")
    print("  step1_collection_overview()     — types and scale")
    print("  step2_inspect_sample_records()  — see ALL metadata fields")
    print("  step3_what_connects_them()      — find overlaps")
    print("  step4_time_periods()            — temporal distribution")
    print("  step5_deep_dive_lodose()        — Lödöse in detail")
    print("  step6_deep_dive_bohuslan()      — Bohusläns in detail")
    print("  step7_toward_the_graph()        — extract data for graph")
    print("  hunt_ksamsok_names()            — find museums in K-samsök")
    print()
    
    # ── Uncomment what you want to run ──
    
    step1_collection_overview()
    step2_inspect_sample_records()
    step3_what_connects_them()
    # step4_time_periods()
    # step5_deep_dive_lodose()
    # step6_deep_dive_bohuslan()
    # step7_toward_the_graph()
    # hunt_ksamsok_names()

╔═════════════════════════════════════════════════════════════╗
║  DigitaltMuseum Explorer                                   ║
║  Bohusläns Museum (S-BM) · Lödöse Museum (S-LM)          ║
╚═════════════════════════════════════════════════════════════╝

Available steps:
  step1_collection_overview()     — types and scale
  step2_inspect_sample_records()  — see ALL metadata fields
  step3_what_connects_them()      — find overlaps
  step4_time_periods()            — temporal distribution
  step5_deep_dive_lodose()        — Lödöse in detail
  step6_deep_dive_bohuslan()      — Bohusläns in detail
  step7_toward_the_graph()        — extract data for graph
  hunt_ksamsok_names()            — find museums in K-samsök

STEP 1: Collection overview

──────────────────────────────────────────────────
  Bohusläns museum (S-BM)
  Total objects:         327,448
  With images:           179,881

  Object types:
    Thing                           175,965  ████████████████████████████████████████
    P

In [12]:
"""
Cross-Museum Connection Explorer
=================================
Follow-up script targeting the specific connections found between
Bohusläns Museum (S-BM) and Lödöse Museum (S-LM):

  1. The 797 Bohusläns objects mentioning "Lödöse"
  2. The medieval intersection (~13k BM + ~15.6k LM)
  3. Material-based connections across time periods
  4. Preparing data for NetworkX graph construction

Prerequisites:
    pip install requests pandas networkx

API key: "demo" returns max 10 rows per query.
For bulk work, contact support@kulturit.no for a full key.
"""

import requests
import json
import time
from collections import Counter, defaultdict
from pprint import pprint

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("(pandas not installed — tabular views disabled)")

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("(networkx not installed — graph building disabled)")


# ============================================================
# CONFIG & CORE
# ============================================================

DIMU_BASE = "https://api.dimu.org/api/solr/select"
API_KEY = "demo"  # Max 10 rows per query


def query(owner, q="*", rows=10, start=0, fq_extra=None,
          facet_field=None, facet_limit=50, sort=None):
    """Flexible DiMu query. Returns parsed JSON."""
    params = {
        "q": q,
        "fq": f"identifier.owner:{owner}*",
        "wt": "json",
        "rows": rows,
        "start": start,
        "api.key": API_KEY,
    }
    if sort:
        params["sort"] = sort
    if facet_field:
        params["facet"] = "true"
        params["facet.field"] = facet_field
        params["facet.limit"] = facet_limit
        params["facet.mincount"] = 1
        params["rows"] = 0

    param_tuples = list(params.items())
    if fq_extra:
        if isinstance(fq_extra, str):
            fq_extra = [fq_extra]
        for fq in fq_extra:
            param_tuples.append(("fq", fq))

    r = requests.get(DIMU_BASE, params=param_tuples, timeout=30)
    r.raise_for_status()
    return r.json()


def get_count(owner, q="*", fq_extra=None):
    result = query(owner, q=q, rows=0, fq_extra=fq_extra)
    return result.get("response", {}).get("numFound", 0)


def get_docs(owner, q="*", n=10, start=0, fq_extra=None, sort=None):
    result = query(owner, q=q, rows=n, start=start,
                   fq_extra=fq_extra, sort=sort)
    return result.get("response", {}).get("docs", [])


def get_facets(owner, field, q="*", limit=50, fq_extra=None):
    result = query(owner, q=q, facet_field=field,
                   facet_limit=limit, fq_extra=fq_extra)
    raw = (result.get("facet_counts", {})
                 .get("facet_fields", {})
                 .get(field, []))
    return [(raw[i], raw[i+1]) for i in range(0, len(raw), 2)]


def extract_record(doc, museum_key):
    """Extract a normalized record dict from a DiMu document."""
    return {
        "museum": museum_key,
        "uuid": doc.get("artifact.uuid", ""),
        "id": doc.get("identifier.id", ""),
        "type": doc.get("artifact.type", ""),
        "title": doc.get("artifact.ingress.title", ""),
        "names": doc.get("artifact.ingress.names", []),
        "description": doc.get("artifact.ingress.description", ""),
        "producer": doc.get("artifact.ingress.producer", ""),
        "producer_role": doc.get("artifact.ingress.producerRole", ""),
        "place": doc.get("artifact.ingress.production.place", ""),
        "from_year": doc.get("artifact.ingress.production.fromYear", None),
        "to_year": doc.get("artifact.ingress.production.toYear", None),
        "subjects": doc.get("artifact.ingress.subjects", []),
        "classification": doc.get("artifact.ingress.classification", []),
        "has_pictures": doc.get("artifact.hasPictures", False),
        "coordinate": doc.get("artifact.coordinate", ""),
        "subtype": doc.get("artifact.subtype", ""),
    }


def print_record(rec, indent=4):
    """Print a normalized record compactly."""
    pfx = " " * indent
    for k, v in rec.items():
        if v and v != [] and v != "":
            val = str(v)[:90]
            print(f"{pfx}{k:18s}: {val}")


# ============================================================
# PART 1: THE 797 — BOHUSLÄNS OBJECTS MENTIONING LÖDÖSE
# ============================================================

def explore_lodose_in_bohuslan():
    """
    What are these ~797 Bohusläns museum objects that mention "Lödöse"?
    Are they photos of the site? Archaeological finds? Documents?
    This is the most explicit bridge between the two collections.
    """
    print("=" * 65)
    print("PART 1: Bohusläns museum objects mentioning 'Lödöse'")
    print("=" * 65)

    total = get_count("S-BM", q="Lödöse")
    print(f"\n  Total Bohusläns objects mentioning 'Lödöse': {total:,}")

    # What types are they?
    print("\n  By object type:")
    types = get_facets("S-BM", "artifact.type", q="Lödöse", limit=10)
    for name, cnt in types:
        print(f"    {name:25s} {cnt:>6,}")

    # What are they about? (subjects)
    print("\n  Top subjects:")
    subjects = get_facets("S-BM", "artifact.ingress.subjects",
                          q="Lödöse", limit=20)
    for name, cnt in subjects:
        print(f"    {name:30s} {cnt:>6,}")

    # Object names/classifications
    print("\n  Top object names:")
    names = get_facets("S-BM", "artifact.name", q="Lödöse", limit=20)
    for name, cnt in names:
        print(f"    {name:30s} {cnt:>6,}")

    # Time distribution
    print("\n  Time periods (fromYear):")
    years = get_facets("S-BM", "artifact.ingress.production.fromYear",
                       q="Lödöse", limit=20)
    for name, cnt in years:
        print(f"    {name:10s} {cnt:>6,}")

    # Sample records
    print("\n  Sample records:")
    docs = get_docs("S-BM", q="Lödöse", n=10)
    for i, doc in enumerate(docs):
        rec = extract_record(doc, "bohuslan")
        print(f"\n  ── BM-Lödöse {i+1} ──")
        print_record(rec)

    return docs


# ============================================================
# PART 2: THE MEDIEVAL INTERSECTION
# ============================================================

def explore_medieval_intersection():
    """
    Both museums have substantial medieval material.
    Bohusläns: ~13k objects matching "medeltid"
    Lödöse: ~15.6k (basically the whole collection)
    
    What does the medieval overlap look like when broken down
    by material, object type, and classification?
    """
    print("\n" + "=" * 65)
    print("PART 2: The medieval intersection")
    print("=" * 65)

    # Counts
    bm_med = get_count("S-BM", q="medeltid")
    lm_med = get_count("S-LM", q="medeltid")
    print(f"\n  Medieval objects:")
    print(f"    Bohusläns museum:  {bm_med:>8,}")
    print(f"    Lödöse museum:     {lm_med:>8,}")

    # Medieval object types — each museum
    print("\n  Medieval object types:")
    print(f"  {'Type':25s} {'Bohusläns':>10s} {'Lödöse':>10s}")
    print(f"  {'─'*25} {'─'*10} {'─'*10}")

    bm_types = dict(get_facets("S-BM", "artifact.type",
                               q="medeltid", limit=10))
    lm_types = dict(get_facets("S-LM", "artifact.type",
                               q="medeltid", limit=10))
    all_types = sorted(set(list(bm_types.keys()) + list(lm_types.keys())))
    for t in all_types:
        bm_c = bm_types.get(t, 0)
        lm_c = lm_types.get(t, 0)
        print(f"  {t:25s} {bm_c:>10,} {lm_c:>10,}")

    # Medieval materials — comparison
    print("\n  Medieval materials:")
    print(f"  {'Material':25s} {'Bohusläns':>10s} {'Lödöse':>10s}")
    print(f"  {'─'*25} {'─'*10} {'─'*10}")

    bm_mat = dict(get_facets("S-BM", "artifact.material",
                             q="medeltid", limit=30))
    lm_mat = dict(get_facets("S-LM", "artifact.material",
                             q="medeltid", limit=30))
    all_mat = sorted(set(list(bm_mat.keys()) + list(lm_mat.keys())))
    for m in all_mat:
        bm_c = bm_mat.get(m, 0)
        lm_c = lm_mat.get(m, 0)
        if bm_c > 0 and lm_c > 0:  # Only shared materials
            print(f"  {m:25s} {bm_c:>10,} {lm_c:>10,}  ◄ shared")
        elif bm_c > 50 or lm_c > 50:  # Or large single-museum counts
            print(f"  {m:25s} {bm_c:>10,} {lm_c:>10,}")

    # Medieval object names — comparison
    print("\n  Medieval object classifications (shared):")
    print(f"  {'Name':25s} {'Bohusläns':>10s} {'Lödöse':>10s}")
    print(f"  {'─'*25} {'─'*10} {'─'*10}")

    bm_names = dict(get_facets("S-BM", "artifact.name",
                               q="medeltid", limit=40))
    lm_names = dict(get_facets("S-LM", "artifact.name",
                               q="medeltid", limit=40))
    shared_names = set(bm_names.keys()) & set(lm_names.keys())
    for n in sorted(shared_names):
        print(f"  {n:25s} {bm_names[n]:>10,} {lm_names[n]:>10,}")

    # Sample medieval records from each
    print("\n  Sample medieval records — Bohusläns museum:")
    bm_docs = get_docs("S-BM", q="medeltid",
                       fq_extra="artifact.type:Thing", n=5)
    for i, doc in enumerate(bm_docs):
        rec = extract_record(doc, "bohuslan")
        print(f"\n  ── BM medieval {i+1} ──")
        print_record(rec)

    print("\n  Sample medieval records — Lödöse museum:")
    lm_docs = get_docs("S-LM", q="medeltid", n=5)
    for i, doc in enumerate(lm_docs):
        rec = extract_record(doc, "lodose")
        print(f"\n  ── LM medieval {i+1} ──")
        print_record(rec)

    return bm_docs, lm_docs


# ============================================================
# PART 3: MATERIAL-BASED CONNECTIONS ACROSS TIME
# ============================================================

def explore_material_connections():
    """
    For each shared material, compare the temporal distribution
    across the two museums. This reveals whether the same types
    of objects from the same era ended up in different collections.
    """
    print("\n" + "=" * 65)
    print("PART 3: Material connections across time")
    print("=" * 65)

    shared_materials = ["keramik", "trä", "järn", "ull", "glas"]

    for material in shared_materials:
        print(f"\n  {'━' * 55}")
        print(f"  Material: {material.upper()}")
        print(f"  {'━' * 55}")

        bm_total = get_count("S-BM", q=material)
        lm_total = get_count("S-LM", q=material)
        print(f"    Bohusläns: {bm_total:>6,}   Lödöse: {lm_total:>6,}")

        # Time distribution for this material in each museum
        print(f"\n    Time distribution (fromYear):")
        print(f"    {'Year':10s} {'Bohusläns':>10s} {'Lödöse':>10s}")
        print(f"    {'─'*10} {'─'*10} {'─'*10}")

        bm_years = dict(get_facets("S-BM",
                                   "artifact.ingress.production.fromYear",
                                   q=material, limit=25))
        lm_years = dict(get_facets("S-LM",
                                   "artifact.ingress.production.fromYear",
                                   q=material, limit=25))

        all_years = sorted(set(list(bm_years.keys()) + list(lm_years.keys())))
        for y in all_years:
            bm_c = bm_years.get(y, 0)
            lm_c = lm_years.get(y, 0)
            marker = " ◄ overlap" if bm_c > 0 and lm_c > 0 else ""
            print(f"    {y:10s} {bm_c:>10,} {lm_c:>10,}{marker}")

        # Object names for this material — what kinds of things?
        print(f"\n    Object types made of {material}:")
        print(f"    {'Bohusläns museum':40s}  {'Lödöse museum':40s}")

        bm_names = get_facets("S-BM", "artifact.name",
                              q=material, limit=10)
        lm_names = get_facets("S-LM", "artifact.name",
                              q=material, limit=10)

        max_rows = max(len(bm_names), len(lm_names))
        for i in range(max_rows):
            bm_str = (f"{bm_names[i][0]:25s} {bm_names[i][1]:>6,}"
                      if i < len(bm_names) else " " * 32)
            lm_str = (f"{lm_names[i][0]:25s} {lm_names[i][1]:>6,}"
                      if i < len(lm_names) else "")
            print(f"      {bm_str}    {lm_str}")

        time.sleep(0.5)


# ============================================================
# PART 4: BUILDING THE CROSS-MUSEUM GRAPH
# ============================================================

def build_connection_graph():
    """
    Build a NetworkX graph from the cross-museum connections.
    
    Strategy: pull medieval objects from both museums and connect
    them through shared materials, time periods, and classifications.
    
    With demo key we get max 10 per query, so we do multiple
    targeted queries to build a richer sample.
    """
    if not HAS_NX:
        print("networkx required: pip install networkx")
        return None

    print("\n" + "=" * 65)
    print("PART 4: Building cross-museum connection graph")
    print("=" * 65)

    G = nx.Graph()

    # Add museum nodes
    G.add_node("Bohusläns museum", node_type="Museum",
               location="Uddevalla", total_objects=327448)
    G.add_node("Lödöse museum", node_type="Museum",
               location="Lödöse", total_objects=20523)

    all_records = []

    # --- Fetch records with targeted queries ---
    fetch_plan = [
        # (museum_code, museum_key, query, description)
        ("S-BM", "bohuslan", "medeltid keramik", "BM medieval ceramics"),
        ("S-BM", "bohuslan", "medeltid järn",    "BM medieval iron"),
        ("S-BM", "bohuslan", "medeltid trä",     "BM medieval wood"),
        ("S-BM", "bohuslan", "medeltid ull",     "BM medieval wool"),
        ("S-BM", "bohuslan", "medeltid ben",     "BM medieval bone"),
        ("S-BM", "bohuslan", "Lödöse",           "BM referencing Lödöse"),
        ("S-LM", "lodose",   "keramik",          "LM ceramics"),
        ("S-LM", "lodose",   "järn",             "LM iron"),
        ("S-LM", "lodose",   "trä",              "LM wood"),
        ("S-LM", "lodose",   "ull",              "LM wool"),
        ("S-LM", "lodose",   "ben",              "LM bone"),
        ("S-LM", "lodose",   "KANNA",            "LM jugs (Hanseatic trade)"),
    ]

    for code, mkey, q, desc in fetch_plan:
        docs = get_docs(code, q=q, n=10)
        records = [extract_record(d, mkey) for d in docs]
        all_records.extend(records)
        print(f"  Fetched {len(records):>2} records: {desc}")
        time.sleep(0.3)

    print(f"\n  Total records fetched: {len(all_records)}")

    # --- Build graph nodes and edges ---
    material_keywords = {
        "keramik", "trä", "järn", "ull", "glas", "ben",
        "brons", "koppar", "sten", "läder", "lin", "silver",
    }

    for rec in all_records:
        obj_id = rec["id"] or rec["uuid"]
        if not obj_id:
            continue

        museum_name = ("Bohusläns museum" if rec["museum"] == "bohuslan"
                       else "Lödöse museum")

        # Object node
        label = rec["title"] or (rec["names"][0] if rec["names"] else obj_id)
        G.add_node(obj_id, node_type="Object", label=label,
                   museum=rec["museum"], title=rec["title"],
                   from_year=rec["from_year"], to_year=rec["to_year"])

        # Museum → Object edge
        G.add_edge(museum_name, obj_id, edge_type="HAS_OBJECT")

        # Time period nodes
        if rec["from_year"]:
            # Create century-level time buckets for easier graph analysis
            century = (int(rec["from_year"]) // 100) * 100
            period_label = f"{century}s"
            G.add_node(period_label, node_type="TimePeriod",
                       century=century)
            G.add_edge(obj_id, period_label, edge_type="DATED_TO",
                       from_year=rec["from_year"],
                       to_year=rec["to_year"])

        # Material nodes — extract from title, names, description
        text_blob = " ".join([
            rec["title"] or "",
            " ".join(rec["names"]) if rec["names"] else "",
            rec["description"] or "",
        ]).lower()

        for mat in material_keywords:
            if mat in text_blob:
                G.add_node(mat, node_type="Material")
                G.add_edge(obj_id, mat, edge_type="MADE_OF")

        # Classification nodes from names
        for name in (rec["names"] or []):
            if name and len(name) > 1:
                name_clean = name.strip().upper()
                G.add_node(name_clean, node_type="Classification")
                G.add_edge(obj_id, name_clean,
                           edge_type="CLASSIFIED_AS")

        # Place nodes
        if rec["place"]:
            # Parse hierarchical place strings
            places = [p.strip() for p in rec["place"].split(",")
                      if p.strip()]
            for place in places:
                G.add_node(place, node_type="Place")
                G.add_edge(obj_id, place, edge_type="LOCATED_AT")

        # Subject nodes
        for subj in (rec["subjects"] or []):
            if subj:
                G.add_node(subj, node_type="Subject")
                G.add_edge(obj_id, subj, edge_type="HAS_SUBJECT")

    # --- Graph analysis ---
    print(f"\n  {'━' * 50}")
    print(f"  Graph summary")
    print(f"  {'━' * 50}")
    print(f"    Nodes: {G.number_of_nodes()}")
    print(f"    Edges: {G.number_of_edges()}")

    node_types = Counter(
        data.get("node_type", "?") for _, data in G.nodes(data=True)
    )
    print(f"\n    Node types:")
    for nt, c in node_types.most_common():
        print(f"      {nt:20s} {c:>5}")

    edge_types = Counter(
        data.get("edge_type", "?") for _, _, data in G.edges(data=True)
    )
    print(f"\n    Edge types:")
    for et, c in edge_types.most_common():
        print(f"      {et:20s} {c:>5}")

    # --- Find cross-museum bridges ---
    print(f"\n  {'━' * 50}")
    print(f"  Cross-museum bridges (shared non-Object nodes)")
    print(f"  {'━' * 50}")

    bridge_nodes = []
    for node, data in G.nodes(data=True):
        if data.get("node_type") in ("Material", "TimePeriod",
                                      "Classification", "Place",
                                      "Subject"):
            neighbors = list(G.neighbors(node))
            museums_connected = set()
            for nb in neighbors:
                nb_data = G.nodes[nb]
                if nb_data.get("node_type") == "Object":
                    museums_connected.add(nb_data.get("museum", "?"))
                elif nb_data.get("node_type") == "Museum":
                    pass  # skip direct museum connections

            if len(museums_connected) > 1:
                bridge_nodes.append({
                    "node": node,
                    "type": data["node_type"],
                    "degree": len(neighbors),
                    "museums": museums_connected,
                })

    if bridge_nodes:
        bridge_nodes.sort(key=lambda x: x["degree"], reverse=True)
        print(f"\n    Found {len(bridge_nodes)} bridge nodes:")
        for b in bridge_nodes[:25]:
            print(f"      [{b['type']:15s}] {b['node']:25s} "
                  f"(degree {b['degree']:>3})")
    else:
        print("\n    No bridge nodes found in this sample.")
        print("    (This likely means the demo key's 10-record limit")
        print("     prevented enough overlap. A full key would fix this.)")

    # --- Highest-degree nodes (most connected) ---
    print(f"\n  {'━' * 50}")
    print(f"  Most connected nodes (top 15)")
    print(f"  {'━' * 50}")

    degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    for node, deg in degrees[:15]:
        data = G.nodes[node]
        ntype = data.get("node_type", "?")
        label = data.get("label", node)
        if ntype == "Object":
            label = f"{label} ({data.get('museum', '?')})"
        print(f"    {deg:>4}  [{ntype:15s}] {label}")

    return G, all_records


# ============================================================
# PART 5: EXPORT FOR FURTHER ANALYSIS
# ============================================================

def export_data(G=None, records=None):
    """
    Export the graph and records for further work in
    notebooks, Neo4j, Gephi, etc.
    """
    print("\n" + "=" * 65)
    print("PART 5: Exporting data")
    print("=" * 65)

    if records and HAS_PANDAS:
        df = pd.DataFrame(records)
        df.to_csv("dimu_cross_museum_records.csv", index=False)
        print(f"  Records → dimu_cross_museum_records.csv ({len(df)} rows)")

        # Summary table
        print(f"\n  Record summary:")
        print(f"    By museum:")
        print(df["museum"].value_counts().to_string(header=False))
        print(f"\n    Fields with data:")
        for col in df.columns:
            filled = df[col].apply(
                lambda x: bool(x) and x != [] and x != ""
            ).sum()
            if filled > 0:
                print(f"      {col:20s} {filled:>4}/{len(df)}")

    if G and HAS_NX:
        # Export as GraphML (readable by Gephi, Neo4j, etc.)
        nx.write_graphml(G, "dimu_cross_museum_graph.graphml")
        print(f"\n  Graph → dimu_cross_museum_graph.graphml")
        print(f"    ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

        # Export as edge list for quick inspection
        with open("dimu_cross_museum_edges.csv", "w") as f:
            f.write("source,target,edge_type\n")
            for u, v, data in G.edges(data=True):
                etype = data.get("edge_type", "")
                # Escape commas in node names
                u_clean = str(u).replace(",", ";")
                v_clean = str(v).replace(",", ";")
                f.write(f"{u_clean},{v_clean},{etype}\n")
        print(f"  Edges → dimu_cross_museum_edges.csv")

        # Export node list
        with open("dimu_cross_museum_nodes.csv", "w") as f:
            f.write("node_id,node_type,label,museum\n")
            for node, data in G.nodes(data=True):
                ntype = data.get("node_type", "")
                label = data.get("label", str(node)).replace(",", ";")
                museum = data.get("museum", "")
                n_clean = str(node).replace(",", ";")
                f.write(f"{n_clean},{ntype},{label},{museum}\n")
        print(f"  Nodes → dimu_cross_museum_nodes.csv")

    print("\n  Next steps:")
    print("    1. Get a full API key from KulturIT (support@kulturit.no)")
    print("    2. Re-run with rows=100 to build a denser graph")
    print("    3. Load .graphml into Gephi for visual exploration")
    print("    4. Or load into Neo4j for Cypher queries")
    print("    5. Run community detection (Louvain) on the graph")
    print("    6. Look for objects that bridge communities")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("╔═════════════════════════════════════════════════════════════╗")
    print("║  Cross-Museum Connection Explorer                          ║")
    print("║  Bohusläns Museum ←→ Lödöse Museum                        ║")
    print("╚═════════════════════════════════════════════════════════════╝")
    print()
    print("Available explorations:")
    print("  explore_lodose_in_bohuslan()    — the 797 BM↔Lödöse objects")
    print("  explore_medieval_intersection() — medieval overlap")
    print("  explore_material_connections()  — materials across time")
    print("  build_connection_graph()        — build NetworkX graph")
    print("  export_data(G, records)         — export for Gephi/Neo4j")
    print()

    # ── Uncomment what you want to run ──

    explore_lodose_in_bohuslan()
    explore_medieval_intersection()
    explore_material_connections()

    # G, records = build_connection_graph()
    # export_data(G, records)

╔═════════════════════════════════════════════════════════════╗
║  Cross-Museum Connection Explorer                          ║
║  Bohusläns Museum ←→ Lödöse Museum                        ║
╚═════════════════════════════════════════════════════════════╝

Available explorations:
  explore_lodose_in_bohuslan()    — the 797 BM↔Lödöse objects
  explore_medieval_intersection() — medieval overlap
  explore_material_connections()  — materials across time
  build_connection_graph()        — build NetworkX graph
  export_data(G, records)         — export for Gephi/Neo4j

PART 1: Bohusläns museum objects mentioning 'Lödöse'

  Total Bohusläns objects mentioning 'Lödöse': 797

  By object type:
    Thing                        608
    Fineart                      110
    Photograph                    77
    Building                       1
    Exhibition                     1

  Top subjects:
    Jordfynd                          338
    Samhälle                          210
    Sjukvård          